In [6]:
import requests
import os
from pathlib import Path
import re
import json
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright
from datetime import datetime
from collections import defaultdict

In [7]:
LEXES_API_URL = "https://polylex-admin.epfl.ch/api/v1/lexes?isAbrogated=0"
DATA_DIR = Path("data")
STATS_DIR = Path("stats")

In [4]:
# retrieve metadata from Polylex API

def retrieve_data(api_url):
    response = requests.get(api_url)
    if response.status_code != 200:
        # TODO : a gerer dans les logs
        raise Exception(f"Unexpected status code while fetching : {response.status_code}")
    data = {
        'pdf': {},
        'fedlex': {},
        'others': {}
    }
    for lex in response.json():
        lex_id = lex.get('_id')
        entries = {
            "fr": {
                "type": lex.get("type"),
                "number": lex.get("number"),
                "url_polylex": lex.get('urlFr'),
                "description": lex.get('descriptionFr')
            },
            "en": {
                "type": lex.get("type"),
                "number": lex.get("number"),
                "url_polylex": lex.get('urlEn'),
                "description": lex.get('descriptionEn')
            },
        }
        for lang, entry in entries.items():
            url_polylex = entry.get('url_polylex')
            if url_polylex.endswith(".pdf"):
                source_format = "pdf"
            elif "www.admin.ch" in url_polylex or "fedlex.admin.ch" in url_polylex:
                source_format = "fedlex"
            else:
                source_format = "others"
            data[source_format].setdefault(lex_id, {})
            data[source_format][lex_id][lang] = entry
    # TODO : a gerer dans les logs
    total_entries = sum(
        len(langs)
        for source_format in data.values()
        for langs in source_format.values()
    )
    assert(total_entries == 2 * len(response.json()))
    return data

data = retrieve_data(LEXES_API_URL)

# add urls to fetch documents from Fedlex

async def resolve_redirects(urls):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        redirects = []
        for url in urls:
            await page.goto(url, wait_until="domcontentloaded")
            await page.wait_for_timeout(1500)
            redirects.append(page.url)
        await browser.close()
        return redirects

def get_fedlex_api_style_url(urls):
    api_urls = []
    for url in urls:
        api_url = url.replace("https://www.fedlex.admin.ch/", "https://fedlex.data.admin.ch/")
        if api_url.endswith("/fr"):
            api_url = api_url.replace("/fr", "")
        api_urls.append(api_url)
    return api_urls

def get_fedlex_pdf(url, lang):
    if lang == "en":
        lang_uri = "ENG"
    else:
        lang_uri = "FRA"
    # TODO : comprendre la requete -> base sur https://fedlex.data.admin.ch/fr-CH/sparql?id=101 puis ChatGPT
    sparql_query = f"""
PREFIX jolux: <http://data.legilux.public.lu/resource/ontology/jolux#>
SELECT ?publicationDate ?dateApplicability ?fileUrl WHERE {{
  ?work jolux:isMemberOf <{url}> ;
        jolux:dateApplicability ?dateApplicability ;
        jolux:isRealizedBy ?expr .
  OPTIONAL {{ ?work jolux:publicationDate ?publicationDate }}
  ?expr jolux:language <http://publications.europa.eu/resource/authority/language/{lang_uri}> ;
        jolux:isEmbodiedBy ?manif .
  ?manif jolux:userFormat <https://fedlex.data.admin.ch/vocabulary/user-format/pdf-a> ;
        jolux:isExemplifiedBy ?fileUrl .
}}
ORDER BY DESC(?publicationDate) DESC(?dateApplicability)
LIMIT 1
""".strip()
    endpoint = "https://fedlex.data.admin.ch/sparqlendpoint"
    r = requests.get(endpoint, params={"query": sparql_query, "format": "application/sparql-results+json"})
    r.raise_for_status()
    data = r.json()
    bindings = data["results"]["bindings"]
    if not bindings:
        print(f"No SPARQL results for {url}")
        return ""
    return bindings[0]["fileUrl"]["value"]

fedlex_jobs = []
for lex_id, lex_metadata_dict in data.get("fedlex", {}).items():
    for lang, lex_metadata in lex_metadata_dict.items():
        url = lex_metadata.get("url_polylex")
        fedlex_jobs.append((lex_id, lang, url))
fedlex_initial_urls = [u for (_, _, u) in fedlex_jobs]
fedlex_redirected_urls = await resolve_redirects(fedlex_initial_urls)
fedlex_sparql_urls = get_fedlex_api_style_url(fedlex_redirected_urls)
for (lex_id, lang, _), sparql_url in zip(fedlex_jobs, fedlex_sparql_urls):
    data["fedlex"][lex_id][lang]["url_sparql"] = sparql_url

# add urls to fetch appendices

def get_appendix_urls(text_with_html):
    soup = BeautifulSoup(text_with_html, "html.parser")
    appendix_urls = []
    for a in soup.find_all("a"):
        href = a.get("href", "").strip()
        if "mailto:" not in href:
            appendix_urls.append(href)
    return appendix_urls

for cat, lexes_metadata_dict in data.items():
    for lex_id, lex_metadata_dict in lexes_metadata_dict.items():
        for lang, lex_metadata in lex_metadata_dict.items():
            appendix_urls = get_appendix_urls(lex_metadata.get('description'))
            data[cat][lex_id][lang]["appendix_urls"] = appendix_urls

In [8]:
# load actual state of metadata in json file

os.makedirs(STATS_DIR, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
metadata_filename = os.path.join(STATS_DIR, f"{timestamp}_lexes_metadata.json")
stats_filename = os.path.join(STATS_DIR, f"{timestamp}_lexes_stats.csv")

with open(metadata_filename, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=4, ensure_ascii=False)

In [9]:
# detect duplication between content to download

def make_ref(cat, lex_id, lex_number, lang):
    return f"{cat}_{lex_id}_{lex_number}_{lang}"

def add_item(items, item_content, item_type, ref):
    items[item_content]["item_types"].add(item_type)
    items[item_content]["refs"].append(ref)

items = defaultdict(lambda: {
    "item_types": set(),
    "refs": [],
})

for cat, lexes_metadata_dict in data.items():
    for lex_id, lex_metadata_dict in lexes_metadata_dict.items():
        for lang, lex_metadata in lex_metadata_dict.items():
            ref = make_ref(cat, lex_id, lex_metadata.get("number"), lang)

            url_polylex = lex_metadata.get("url_polylex")
            add_item(items, url_polylex, "url_polylex", ref)

            desc = lex_metadata.get("description")
            add_item(items, desc, "description", ref)

            appendix_urls = lex_metadata.get("appendix_urls")
            for u in appendix_urls:
                add_item(items, u, "appendix_url", ref)

            url_sparql = lex_metadata.get("url_sparql")
            if url_sparql is not None:
                add_item(items, url_sparql, "url_sparql", ref)

duplications = {}
for item, info in items.items():
    if len(info["refs"]) > 1:
        duplications[item] = {
            "item_types": sorted(info["item_types"]),
            "refs": sorted(info["refs"])
        }

In [10]:
# TODO : comment garder le lien entre document et les differentes sources, dans nom du fichier ou dans metadonnees ?

for item, item_info in duplications.items():
    print(f"{len(item_info["refs"])} duplications for {item}")

#duplications

3 duplications for https://www.epfl.ch/about/vice-presidencies/fr/vice-presidence-academique-vpa/textes-de-reference-vpa/
3 duplications for https://www.epfl.ch/about/vice-presidencies/vice-presidency-for-academic-affairs-vpa/reference-texts-vpa/
2 duplications for https://www.epfl.ch/about/overview/wp-content/uploads/2019/09/1.1.7-Ruling-EPFL-Geneve.pdf
4 duplications for <p>Exonération de l'Ecole polytechnique fédérale de Lausanne (EPFL) en matière d'impôts sur les successions et sur les donations.</p>
2 duplications for <p>Exemption of the Ecole polytechnique fédérale de Lausanne (EPFL) from inheritance and gift tax.</p>
2 duplications for https://www.epfl.ch/about/overview/wp-content/uploads/2019/09/1.1.7-Ruling-EPFL-Vaud-.pdf
2 duplications for http://www.admin.ch/ch/f/rs/c172_010_21.html
2 duplications for https://ethrat.stage.mxm.agency/wp-content/uploads/2021/10/Reglement_comites_CEPF_0.pdf
2 duplications for https://ethrat.stage.mxm.agency/wp-content/uploads/2021/11/Directives

In [11]:
# filter metadata to keep only content we want to (based on some rules)

# TODO : definir les regles de filtrage

In [17]:
# load all documents

def create_filename(id, lang, type, number, format, item_type=""):
    # TODO : definir un id unique et creer un fichier avec les reference(s) pour chaque fichier
    item_type_in_name = "_" + item_type if item_type != "" else ""
    return lang + "_" + id + "_" + type + "_" + number + item_type_in_name + f".{format}"

def download_pdf(url, dir, filename):
    headers = {
        "User-Agent": "Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:147.0) Gecko/20100101 Firefox/147.0"
    }
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        with open(os.path.join(dir, filename), "wb") as f:
            f.write(response.content)
    else:
        print(f"Error: the content for {url} can not be fetched (status: {response.status_code})")

def transform_html_in_text(text_with_html):
    soup = BeautifulSoup(text_with_html, "html.parser")
    for a in soup.find_all("a"):
        href = a.get("href", "").strip()
        label = a.get_text()
        a.replace_with(f"{label} ({href})" if href else label)
    for br in soup.find_all("br"):
        br.replace_with("\n")
    text = soup.get_text()
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# TODO : finir d'implementer le loading selon refactoring
# si type == description alors ecrire le contenu directement
# sinon si url (url_polylex, appendix_url ou url_sparql)
    # si url_sparql != None ou si appendix_url et correspond a regexp alors telecharger selon differentes fonctions pour obtenir pdf
    # sinon
        # si url contient inside alors message d'avertissement et rien
        # sinon si url finit par pdf ou docx alors load
        # sinon surement un site et donc message d'avertissement et rien

os.makedirs(DATA_DIR, exist_ok=True)

# TODO : pdfs viennent de https://www.epfl.ch/about/overview/wp-content/uploads, https://ethrat.stage.mxm.agency/wp-content/uploads, https://ethrat.ch/wp-content/uploads et https://www.edoeb.admin.ch/dam/fr/sd-web/jYpssE3v3PL4/leitfaden_internet-unde-mailueberwachungamarbeitsplatzprivatwirt_FR.pdf
# pdfs
for lex_id, langs in data.get("pdf", {}).items():
    for lang, metadata in langs.items():
        url = metadata.get("url_polylex")
        filename = create_filename(lex_id, lang, metadata.get('type'), metadata.get('number'), "pdf")
        download_pdf(url, DATA_DIR, filename)

# Fedlex
for lex_id, langs in data.get("fedlex", {}).items():
    for lang, metadata in langs.items():
        url = metadata.get("url_sparql")
        filename = create_filename(lex_id, lang, metadata.get('type'), metadata.get('number'), "pdf")
        pdf_url = get_fedlex_pdf(url, lang)
        if pdf_url != "":
            download_pdf(pdf_url, DATA_DIR, filename)
        # TODO : gerer les cas ou on ne recupere pas les pdfs car ko

# others
# TODO (voir ci-dessous)
# https://www.efv.admin.ch/efv/fr/home/themen/finanzpolitik_grundlagen/risiko_versicherungspolitik.html -> site de la confederation avec texte de presentation et pdfs a telecharger
# https://www.epfl.ch/about/overview/fr/reglements-et-directives/compliance/compliance-guide/compliance-guide-2025 -> site de l'epfl avec plein de liens vers d'autres textes (possible de scrapper)
# https://www.epfl.ch/education/phd/fr/reglements -> site de l'epfl avec des liens qui menent vers d'autres liens (fedlex, pdfs, ...)
# http://isa.epfl.ch/imoniteur_ISAP/%21gedpublicreports.htm?ww_i_reportmodel=1715636965 -> pas utile, si ?
# https://www.epfl.ch/education/studies/reglement-et-procedure/plans_etudes -> page avec plein de liens (duplique 3 fois)

#data['others']

No SPARQL results for https://fedlex.data.admin.ch/eli/cc/1993/210_210_210/en
No SPARQL results for https://fedlex.data.admin.ch/eli/cc/2004/16
No SPARQL results for https://fedlex.data.admin.ch/eli/cc/2004/17
No SPARQL results for https://fedlex.data.admin.ch/eli/cc/1967/1505_1553_1547
No SPARQL results for https://fedlex.data.admin.ch/eli/cc/2004/179
No SPARQL results for https://fedlex.data.admin.ch/eli/cc/2025/782
No SPARQL results for https://fedlex.data.admin.ch/eli/oc/2025/607
No SPARQL results for https://fedlex.data.admin.ch/eli/oc/2025/607
No SPARQL results for https://fedlex.data.admin.ch/eli/cc/1995/3853_3853_3853
No SPARQL results for https://fedlex.data.admin.ch/eli/cc/2014/525
No SPARQL results for https://fedlex.data.admin.ch/eli/cc/2001/123
No SPARQL results for https://fedlex.data.admin.ch/eli/cc/2001/124
No SPARQL results for https://fedlex.data.admin.ch/eli/cc/2001/279
No SPARQL results for https://fedlex.data.admin.ch/eli/cc/2012/191
No SPARQL results for https://f

In [12]:
# TODO

#filename = create_filename(lex_id, lang, lex_metadata.get('type'), lex_metadata.get('number'), "txt", "description")
            #with open(os.path.join(DATA_DIR, filename, encoding="utf-8"), "w") as f:
            #    f.write(desc_content)

In [13]:
# TODO : a deplacer dans stats

print(f"Categories of data ({len(data)} categories):")

for cat, lexes_metadata_dict in data.items():
    nb_items = 0
    for lex_id, lex_metadata_dict in lexes_metadata_dict.items():
        for lang, lex_metadata in lex_metadata_dict.items():
            nb_items += 1
    print(f" - '{cat}': {len(data[cat].keys())} items (each one in one or several languages, in total {nb_items} items)")

first_id_in_pdf, first_lex_value_in_pdf = next(iter(data['pdf'].items()))
print(f"Languages of data ({len(data['pdf'][first_id_in_pdf])} languages): {list(data['pdf'][first_id_in_pdf].keys())}")
print(f"Example of content of an item in 'pdf' categorie: {next(iter(data['pdf'].items()))}")

Categories of data (3 categories):
 - 'pdf': 135 items (each one in one or several languages, in total 265 items)
 - 'fedlex': 32 items (each one in one or several languages, in total 59 items)
 - 'others': 7 items (each one in one or several languages, in total 14 items)
Languages of data (2 languages): ['fr', 'en']
Example of content of an item in 'pdf' categorie: ('nYfecNfg8Y8RJazrW', {'fr': {'type': 'LEX', 'number': '1.1.1', 'url_polylex': 'https://www.epfl.ch/about/overview/wp-content/uploads/2019/09/1.1.1_o_organisation_EPFL_fr.pdf', 'description': '<p>Cette ordonnance définit la structure de l’Ecole polytechnique fédérale de Lausanne (EPFL), l’organisation et les compétences décisionnelles de sa Direction, de ses facultés, de ses collèges et de ses organes centraux.&nbsp;</p>\n<p><br>\nL’utilisation du <a href="https://www.epfl.ch/campus/services/communication/identite-visuelle/logo">logo de l’EPFL</a>&nbsp; doit suivre les règles définies par <a href="https://mediacom.epfl.ch">

In [14]:
# TODO : exemple pour iterer sur la structure

for cat, lexes_metadata_dict in data.items():
    for lex_id, lex_metadata_dict in lexes_metadata_dict.items():
        for lang, lex_metadata in lex_metadata_dict.items():
            print(cat)
            print(lex_id)
            print(lang)
            print(lex_metadata)
            break
        break
    break

pdf
nYfecNfg8Y8RJazrW
fr
{'type': 'LEX', 'number': '1.1.1', 'url_polylex': 'https://www.epfl.ch/about/overview/wp-content/uploads/2019/09/1.1.1_o_organisation_EPFL_fr.pdf', 'description': '<p>Cette ordonnance définit la structure de l’Ecole polytechnique fédérale de Lausanne (EPFL), l’organisation et les compétences décisionnelles de sa Direction, de ses facultés, de ses collèges et de ses organes centraux.&nbsp;</p>\n<p><br>\nL’utilisation du <a href="https://www.epfl.ch/campus/services/communication/identite-visuelle/logo">logo de l’EPFL</a>&nbsp; doit suivre les règles définies par <a href="https://mediacom.epfl.ch">Mediacom</a>. Il inclut impérativement le sigle et la dénomination en toutes lettres, placée en dessous de celui-ci. Ces deux éléments sont indissociables. A l’exception d’unités ou de projets qui découlent de partenariats externes, le logo de l’EPFL est le seul logo officiel au sein de l’institution. On ne peut lui adjoindre un autre logo spécifique à une unité.</p>\n<p

In [15]:
# TODO : a supprimer

nb_lex_from_fedlex_with_0_in_number = 0

nb_lex_from_fedlex = 0
for lex_id, lex_metadata_dict in data['fedlex'].items():
    for lang, lex_metadata in lex_metadata_dict.items():
        nb_lex_from_fedlex += 1

for lex_id, lex_metadata_dict in data['fedlex'].items():
    for lang, lex_metadata in lex_metadata_dict.items():
        if ".0." in lex_metadata.get("number"):
            nb_lex_from_fedlex_with_0_in_number += 1
        else:
            print(f"Exception for {lex_metadata.get('number')} in {lang} (id: {lex_id})")

print(f"Result: {nb_lex_from_fedlex_with_0_in_number} / {nb_lex_from_fedlex} lexes fetching Fedlex have a '0' in the lex number")

Exception for 4.1.10 in fr (id: AZb7SPbDfoYbd3p3h)
Exception for 4.1.10 in en (id: AZb7SPbDfoYbd3p3h)
Result: 57 / 59 lexes fetching Fedlex have a '0' in the lex number
